In [3]:

"""
ETAPA 1: Acesso à Imagem Sentinel-2 para o sudoeste do Pará
Implementação robusta garantindo que o ROI fique totalmente dentro do estado
"""

import ee
import geemap
import json

ee.Initialize(project="desafio-solved")

municipios = ee.FeatureCollection("FAO/GAUL/2015/level2")

lista_municipios = [
    'Altamira', 'Anapu', 'Brasil Novo', 'Medicilandia',
    'Pacaja', 'Senador Jose Porfirio', 'Uruara', 'Vitoria do Xingu',
    'Itaituba', 'Aveiro', 'Jacareacanga', 'Novo Progresso',
    'Ruropolis', 'Trairao'
]

sudoeste = municipios.filter(ee.Filter.inList("ADM2_NAME", lista_municipios))

roi = sudoeste.geometry()

colecao = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

imagem = colecao.median().clip(roi)

mapa = geemap.Map()
mapa.centerObject(roi, 7)

mapa.addLayer(imagem, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}, 'Sudoeste do Pará')

mapa


Map(center=[-5.915113179729727, -54.909147137270836], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
"""
ETAPA 2:
"""

import ee
import geemap

def mask_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))

    return image.updateMask(mask)

colecao_limpa = colecao.map(mask_clouds)

imagem = colecao_limpa.median().clip(roi)

ndvi = imagem.normalizedDifference(['B8', 'B4']).rename('NDVI')

mapa.addLayer(ndvi, {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'green']
}, 'NDVI')

vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 2500,  
    'gamma': 1.3
}

mapa

AttributeError: 'ImageCollection' object has no attribute 'normalizedDifference'

Ótima pergunta — agora você entrou na parte que transforma a imagem em algo útil pra análise 👏

Vou te explicar isso de forma bem simples: o que é, pra que serve e como fazer no código.

🧠 🎯 O que é “pré-processamento”?

É basicamente:

“melhorar a imagem antes de analisar”

Pensa como editar uma foto antes de usar:

tirar partes ruins (nuvens ☁️)
melhorar qualidade
destacar informações (vegetação 🌱)
📦 No seu caso (Sentinel-2 no Google Earth Engine)

Você já está usando:

'COPERNICUS/S2_SR_HARMONIZED'

👉 Isso já significa:
✔️ imagem com correção atmosférica aplicada

Então você NÃO precisa se preocupar com isso ✅

🪜 Agora vamos para o que você pode fazer (de verdade)
🟢 1. Remoção de nuvens (mais importante)

Mesmo usando filtro de 20%, ainda sobra nuvem.

💡 Ideia:

Remover pixels com nuvem usando uma máscara.

Código simples:
def mask_clouds(image):
    qa = image.select('QA60')  # banda de qualidade
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))

    return image.updateMask(mask)

colecao_limpa = colecao.map(mask_clouds)

👉 Isso remove nuvens pixel por pixel (bem melhor que só filtro)

🟡 2. Criar índice de vegetação (NDVI)

Esse é MUITO usado 🌱

🧠 O que é NDVI?

Ele mostra:

verde forte → vegetação saudável
marrom → solo ou desmatamento
💻 Código:
ndvi = imagem.normalizedDifference(['B8', 'B4']).rename('NDVI')

👉 Explicando:

B8 = infravermelho (vegetação reflete muito)
B4 = vermelho
Mostrar no mapa:
mapa.addLayer(ndvi, {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'green']
}, 'NDVI')
🔵 3. Melhorar a imagem (visual)

Você pode ajustar contraste:

vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 2500,  # menor = mais contraste
    'gamma': 1.3
}

👉 Isso deixa a imagem mais “bonita”

🧩 Como juntar tudo (fluxo ideal)

Ordem correta:

# 1. remover nuvens
colecao_limpa = colecao.map(mask_clouds)

# 2. criar imagem
imagem = colecao_limpa.median().clip(roi)

# 3. NDVI
ndvi = imagem.normalizedDifference(['B8', 'B4'])
⚠️ O que é opcional vs essencial?
Etapa	Importância
Filtro de nuvem (20%)	✅ essencial
Máscara de nuvem	⭐ recomendado
NDVI	⭐ recomendado
Correção atmosférica	❌ já está pronta
🧠 📌 Resumo simples

Pré-processamento = melhorar a imagem antes de usar

Você pode:

☁️ remover nuvens melhor
🌱 destacar vegetação (NDVI)
🎨 melhorar visual
💬 Em linguagem bem direta

👉 Antes:
“imagem bruta com ruído”

👉 Depois:
“imagem limpa + pronta pra análise”

🚀 Próximo passo natural

Depois disso você já pode:

classificar uso do solo 🗺️
detectar desmatamento 🌳
separar água, floresta, cidade

Se quiser, posso te montar um código final completo (limpo + NDVI + mapa) já pronto pra usar 👍